# CV Task 09: Object Detection Using CNN Architecture (YOLOv11)

**Course:** Computer Vision (CV)  
**Task:** Task 1 — End-to-End Object Detection Using YOLOv11  
**Dataset:** CPPE-5 (Medical Personal Protective Equipment — 5 Classes: `Coverall`, `Face_Shield`, `Gloves`, `Goggles`, `Mask`)  
**Runtime:** Google Colab (Recommended: **Runtime -> Change runtime type -> T4 GPU**)

---

## Notebook Roadmap (Deliverables Checklist)
1. **Environment Setup & GPU Verification** — Install Ultralytics YOLOv11 & dependencies, verify CUDA GPU.
2. **Automatic Dataset Fetching** — Download `rishitdagli/cppe-5` directly from Hugging Face (no API keys required) and inspect raw COCO annotations.
3. **Data Preprocessing & YOLO Formatting** — Aspect-ratio-preserving letterbox resizing (`640×640`), pixel normalization analysis (`[0, 1]`), COCO-to-YOLO bounding-box conversion with strict clamping, and **75% Train / 15% Val / 10% Unseen Test** splitting + `data.yaml`.
4. **YOLOv11 Architecture & Model Setup** — Architectural breakdown (C3k2, SPPF, C2PSA, PAN-FPN, Anchor-Free Decoupled Head) and loading pretrained `yolo11n.pt`.
5. **Model Training** — Fine-tuning YOLOv11 on the prepared dataset with full loss/metric tracking.
6. **Comprehensive Evaluation** — Computing **mAP@0.50**, **mAP@0.50:0.95**, **Precision/Recall/F1**, exact **Mean IoU (Intersection over Union)**, and **Inference Speed (ms/frame & FPS)**.
7. **Testing on Unseen Data** — Generalization testing on held-out test images with Ground Truth vs. Prediction visualizations.
8. **Live Prediction on Frames** — Three ready-to-use live inference modes: (A) Real-time video stream processing with FPS overlay, (B) Interactive Google Colab live webcam frame detection, and (C) Exportable local OpenCV webcam script (`live_webcam_yolo11.py`).
9. **Export Deliverables** — Auto-generated Markdown report populated with your run's exact metrics + downloadable `best.pt` weights and submission archive.

## 1. Environment Setup & Hardware Verification
Make sure GPU acceleration is enabled in Google Colab (**Runtime → Change runtime type → T4 GPU**). This cell installs the latest `ultralytics` package (for YOLOv11) along with Hugging Face `datasets` and visualization libraries.

In [ ]:
# Install required packages cleanly in Google Colab
!pip install -q ultralytics datasets opencv-python-headless matplotlib seaborn pandas scikit-learn pillow

import os
import sys
import time
import glob
import yaml
import json
import shutil
import random
from pathlib import Path

import cv2
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image
from sklearn.model_selection import train_test_split
from datasets import load_dataset
import ultralytics
from ultralytics import YOLO

# Disable Ultralytics telemetry prompts
!yolo settings sync=False > /dev/null 2>&1

# Set global random seeds for reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Detect hardware
DEVICE = 0 if torch.cuda.is_available() else "cpu"
print(f"Python version     : {sys.version.split()[0]}")
print(f"PyTorch version    : {torch.__version__}")
print(f"Ultralytics version: {ultralytics.__version__}")
print(f"CUDA available     : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device         : {torch.cuda.get_device_name(0)}")
else:
    print("WARNING: Running on CPU. Switch to Colab GPU (Runtime -> Change runtime type -> T4 GPU) for fast training!")

## 2. Dataset Acquisition & Exploration
We automatically fetch the **CPPE-5 (Medical Personal Protective Equipment)** object detection dataset (`rishitdagli/cppe-5`) directly via Hugging Face `datasets`.

* **Why this dataset?** It requires zero API keys, contains real-world multi-object scenes with overlapping items and varying scales, and ships in raw COCO format (`[x_min, y_min, width, height]` in pixels)—requiring genuine preprocessing, coordinate transformation, normalization, and train/val/test splitting for YOLOv11.
* **Classes (5):** `0: Coverall`, `1: Face_Shield`, `2: Gloves`, `3: Goggles`, `4: Mask`.

In [ ]:
# Fetch the CPPE-5 dataset automatically from Hugging Face
print("Downloading CPPE-5 dataset from Hugging Face...")
raw_dataset = load_dataset("rishitdagli/cppe-5")
print(raw_dataset)

# Extract class names directly from dataset metadata
CLASS_NAMES = raw_dataset["train"].features["objects"].feature["category"].names
NUM_CLASSES = len(CLASS_NAMES)
print(f"\nNumber of classes: {NUM_CLASSES}")
for idx, name in enumerate(CLASS_NAMES):
    print(f"  Class {idx}: {name}")

# Combine official train and test pools so we can build a balanced, clean 75% / 15% / 10% split
all_samples = []
for split_name in raw_dataset.keys():
    for item in raw_dataset[split_name]:
        all_samples.append(item)

print(f"\nTotal images fetched: {len(all_samples)}")

# Inspect one sample's raw structure
sample_0 = all_samples[0]
print("Sample keys:", list(sample_0.keys()))
print("Raw image size (W, H):", sample_0["image"].size)
print("Number of objects in sample 0:", len(sample_0["objects"]["bbox"]))
print("Example raw COCO bbox [x_min, y_min, w, h]:", sample_0["objects"]["bbox"][0])
print("Example category ID:", sample_0["objects"]["category"][0], "->", CLASS_NAMES[sample_0["objects"]["category"][0]])

In [ ]:
# Visualize 4 raw images with original COCO bounding boxes before preprocessing
COLORS = plt.cm.tab10(np.linspace(0, 1, NUM_CLASSES))

fig, axes = plt.subplots(1, 4, figsize=(20, 5))
for ax, sample in zip(axes, all_samples[:4]):
    img = sample["image"].convert("RGB")
    ax.imshow(img)
    bboxes = sample["objects"]["bbox"]
    cats = sample["objects"]["category"]
    for bbox, cat in zip(bboxes, cats):
        x, y, w, h = bbox
        color = COLORS[cat]
        rect = patches.Rectangle((x, y), w, h, linewidth=2, edgecolor=color, facecolor="none")
        ax.add_patch(rect)
        ax.text(
            x, max(0, y - 4), CLASS_NAMES[cat],
            color="white", fontsize=8, fontweight="bold",
            bbox=dict(facecolor=color, alpha=0.8, pad=1, edgecolor="none")
        )
    ax.set_title(f"Raw Size: {img.size[0]}x{img.size[1]} | {len(bboxes)} objs", fontsize=10)
    ax.axis("off")

plt.suptitle("Raw Dataset Samples (Original Resolutions & COCO Pixel Bounding Boxes)", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

## 3. Data Preprocessing, Dataset Splitting & YOLO Format Preparation

To satisfy all preprocessing requirements cleanly and prevent object aspect-ratio distortion:
1. **Stratified-Aware Train / Val / Test Partitioning:** Split the full dataset into **75% Train**, **15% Validation**, and **10% Unseen Test** (`random_state=42`). Every image belongs to strictly one split to prevent data leakage.
2. **Aspect-Ratio Preserving Letterbox Resize (`640×640`):** Resize each RGB image so its longest side fits `640 px`, then pad symmetrically with neutral gray `(114, 114, 114)` to exact `640×640` dimensions.
3. **Bounding-Box Transformation & YOLO Normalization:** Map every COCO bounding box `[x_min, y_min, w, h]` into the resized & padded `640×640` frame, clip to valid bounds `[0, 640]`, discard degenerate boxes (`w <= 1` or `h <= 1`), and convert to normalized YOLO format:
   $$x_{\text{center}} = \frac{x_1 + x_2}{2W}, \quad y_{\text{center}} = \frac{y_1 + y_2}{2H}, \quad w_{\text{norm}} = \frac{x_2 - x_1}{W}, \quad h_{\text{norm}} = \frac{y_2 - y_1}{H}$$
4. **Pixel Normalization Verification:** Compute and verify `[0, 255]` → `[0.0, 1.0]` normalized pixel statistics across the training set (Ultralytics automatically divides uint8 image tensors by `255.0` on-the-fly during DataLoader collation).
5. **Configuration Generation:** Write `data.yaml` pointing to `train`, `val`, and `test` image directories.

In [ ]:
# Configuration for preprocessing and dataset splits
TARGET_SIZE = 640
DATASET_DIR = Path("/content/cppe5_yolo") if Path("/content").exists() else Path("./cppe5_yolo")

# Clean existing directory if re-running cell
if DATASET_DIR.exists():
    shutil.rmtree(DATASET_DIR)

for split in ["train", "val", "test"]:
    (DATASET_DIR / "images" / split).mkdir(parents=True, exist_ok=True)
    (DATASET_DIR / "labels" / split).mkdir(parents=True, exist_ok=True)


def letterbox_resize(img_rgb: np.ndarray, target_size: int = 640, pad_color=(114, 114, 114)):
    """
    Resize an RGB image to (target_size, target_size) while preserving aspect ratio
    using symmetric letterbox padding. Returns resized image, scale factor, and (pad_x, pad_y).
    """
    orig_h, orig_w = img_rgb.shape[:2]
    scale = min(target_size / orig_w, target_size / orig_h)
    new_w = int(round(orig_w * scale))
    new_h = int(round(orig_h * scale))

    resized = cv2.resize(img_rgb, (new_w, new_h), interpolation=cv2.INTER_LINEAR)

    pad_w = target_size - new_w
    pad_h = target_size - new_h
    pad_left = pad_w // 2
    pad_right = pad_w - pad_left
    pad_top = pad_h // 2
    pad_bottom = pad_h - pad_top

    padded = cv2.copyMakeBorder(
        resized, pad_top, pad_bottom, pad_left, pad_right,
        borderType=cv2.BORDER_CONSTANT, value=pad_color
    )
    return padded, scale, pad_left, pad_top


def coco_to_yolo_letterboxed(bbox, orig_w, orig_h, scale, pad_left, pad_top, target_size=640):
    """
    Convert a raw COCO bbox [x_min, y_min, w, h] into normalized YOLO [xc, yc, wn, hn]
    in the letterboxed target_size x target_size coordinate system.
    Returns None if the bounding box is invalid or degenerate.
    """
    x_min, y_min, bw, bh = bbox
    if bw <= 0 or bh <= 0:
        return None

    # Clamp original box to original image boundaries
    x1 = max(0.0, min(float(x_min), float(orig_w)))
    y1 = max(0.0, min(float(y_min), float(orig_h)))
    x2 = max(0.0, min(float(x_min + bw), float(orig_w)))
    y2 = max(0.0, min(float(y_min + bh), float(orig_h)))

    if (x2 - x1) <= 1.0 or (y2 - y1) <= 1.0:
        return None

    # Transform to letterboxed coordinates
    nx1 = max(0.0, min(x1 * scale + pad_left, float(target_size)))
    ny1 = max(0.0, min(y1 * scale + pad_top, float(target_size)))
    nx2 = max(0.0, min(x2 * scale + pad_left, float(target_size)))
    ny2 = max(0.0, min(y2 * scale + pad_top, float(target_size)))

    nbw = nx2 - nx1
    nbh = ny2 - ny1
    if nbw <= 1.0 or nbh <= 1.0:
        return None

    # Normalize to [0, 1]
    xc = ((nx1 + nx2) / 2.0) / target_size
    yc = ((ny1 + ny2) / 2.0) / target_size
    wn = nbw / target_size
    hn = nbh / target_size

    # Final safety clamp
    xc = max(0.0, min(1.0, xc))
    yc = max(0.0, min(1.0, yc))
    wn = max(0.0, min(1.0, wn))
    hn = max(0.0, min(1.0, hn))

    return xc, yc, wn, hn


# Determine primary class per image for stratified splitting
valid_samples = []
primary_classes = []
for sample in all_samples:
    cats = sample["objects"]["category"]
    bboxes = sample["objects"]["bbox"]
    if len(cats) > 0 and len(bboxes) > 0:
        valid_samples.append(sample)
        # Use rarest/first class present as stratification key
        primary_classes.append(cats[0])

# Split: 75% Train, 15% Val, 10% Test (Unseen Data)
indices = np.arange(len(valid_samples))
train_idx, temp_idx = train_test_split(
    indices, test_size=0.25, random_state=SEED, stratify=primary_classes
)
temp_labels = [primary_classes[i] for i in temp_idx]
val_idx, test_idx = train_test_split(
    temp_idx, test_size=0.40, random_state=SEED, stratify=temp_labels  # 0.40 of 25% = 10% total
)

splits = {
    "train": train_idx,
    "val": val_idx,
    "test": test_idx
}

split_stats = {s: {"images": 0, "boxes": 0, "dropped_boxes": 0, "class_counts": {c: 0 for c in CLASS_NAMES}} for s in splits}
norm_channel_means = []
norm_channel_stds = []

print("Preprocessing images (RGB -> Letterbox 640x640 -> YOLO normalized labels)...")
for split_name, idx_list in splits.items():
    for count, idx in enumerate(idx_list):
        sample = valid_samples[idx]
        pil_img = sample["image"].convert("RGB")
        orig_w, orig_h = pil_img.size
        img_np = np.array(pil_img)

        # 1. Letterbox resize to 640x640
        resized_rgb, scale, pad_left, pad_top = letterbox_resize(img_np, TARGET_SIZE)

        # 2. Compute normalized [0, 1] pixel stats on training split for verification
        if split_name == "train":
            img_norm = resized_rgb.astype(np.float32) / 255.0
            norm_channel_means.append(img_norm.mean(axis=(0, 1)))
            norm_channel_stds.append(img_norm.std(axis=(0, 1)))

        # 3. Convert bounding boxes to normalized YOLO format
        yolo_lines = []
        for bbox, cat in zip(sample["objects"]["bbox"], sample["objects"]["category"]):
            converted = coco_to_yolo_letterboxed(bbox, orig_w, orig_h, scale, pad_left, pad_top, TARGET_SIZE)
            if converted is None:
                split_stats[split_name]["dropped_boxes"] += 1
                continue
            xc, yc, wn, hn = converted
            yolo_lines.append(f"{cat} {xc:.6f} {yc:.6f} {wn:.6f} {hn:.6f}")
            split_stats[split_name]["boxes"] += 1
            split_stats[split_name]["class_counts"][CLASS_NAMES[cat]] += 1

        if not yolo_lines:
            continue

        # 4. Save preprocessed image & YOLO label file
        stem = f"{split_name}_{count:04d}"
        img_out_path = DATASET_DIR / "images" / split_name / f"{stem}.jpg"
        lbl_out_path = DATASET_DIR / "labels" / split_name / f"{stem}.txt"

        cv2.imwrite(str(img_out_path), cv2.cvtColor(resized_rgb, cv2.COLOR_RGB2BGR), [int(cv2.IMWRITE_JPEG_QUALITY), 95])
        with open(lbl_out_path, "w") as f:
            f.write("\n".join(yolo_lines) + "\n")

        split_stats[split_name]["images"] += 1

# Generate data.yaml required by Ultralytics YOLOv11
data_yaml = {
    "path": str(DATASET_DIR.resolve()),
    "train": "images/train",
    "val": "images/val",
    "test": "images/test",
    "nc": NUM_CLASSES,
    "names": {i: name for i, name in enumerate(CLASS_NAMES)}
}

YAML_PATH = DATASET_DIR / "data.yaml"
with open(YAML_PATH, "w") as f:
    yaml.dump(data_yaml, f, sort_keys=False)

print("\nPreprocessing Complete! Dataset Split Summary:")
summary_df = pd.DataFrame([
    {
        "Split": s.upper(),
        "Images": split_stats[s]["images"],
        "Total Boxes": split_stats[s]["boxes"],
        "Avg Boxes/Img": round(split_stats[s]["boxes"] / max(1, split_stats[s]["images"]), 2),
        "Invalid Boxes Dropped": split_stats[s]["dropped_boxes"]
    }
    for s in ["train", "val", "test"]
])
display(summary_df)

mean_rgb = np.mean(norm_channel_means, axis=0)
std_rgb = np.mean(norm_channel_stds, axis=0)
print(f"\nNormalized Image Pixel Verification ([0.0, 1.0] scale):")
print(f"  Train RGB Mean : R={mean_rgb[0]:.4f}, G={mean_rgb[1]:.4f}, B={mean_rgb[2]:.4f}")
print(f"  Train RGB Std  : R={std_rgb[0]:.4f}, G={std_rgb[1]:.4f}, B={std_rgb[2]:.4f}")
print(f"\nGenerated YOLO config file: {YAML_PATH}")
print(YAML_PATH.read_text())

In [ ]:
# Sanity-check: Visualize preprocessed 640x640 images with reconstructed YOLO annotations + class distribution
fig, axes = plt.subplots(1, 4, figsize=(20, 5))
sample_train_imgs = sorted(glob.glob(str(DATASET_DIR / "images/train/*.jpg")))[:4]

for ax, img_path in zip(axes, sample_train_imgs):
    img_bgr = cv2.imread(img_path)
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    h, w = img_rgb.shape[:2]
    lbl_path = img_path.replace("/images/", "/labels/").replace(".jpg", ".txt")

    ax.imshow(img_rgb)
    if os.path.exists(lbl_path):
        with open(lbl_path, "r") as f:
            for line in f:
                cls_id, xc, yc, bw, bh = map(float, line.strip().split())
                cls_id = int(cls_id)
                x1 = (xc - bw / 2.0) * w
                y1 = (yc - bh / 2.0) * h
                rect = patches.Rectangle(
                    (x1, y1), bw * w, bh * h,
                    linewidth=2, edgecolor=COLORS[cls_id], facecolor="none"
                )
                ax.add_patch(rect)
                ax.text(
                    x1, max(0, y1 - 4), CLASS_NAMES[cls_id],
                    color="white", fontsize=8, fontweight="bold",
                    bbox=dict(facecolor=COLORS[cls_id], alpha=0.85, pad=1, edgecolor="none")
                )
    ax.set_title(f"{Path(img_path).name} ({w}x{h})", fontsize=10)
    ax.axis("off")

plt.suptitle("Preprocessed 640×640 Letterboxed Images with Verified Normalized YOLO Labels", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

# Plot class distribution across Train, Val, and Test splits
class_dist_df = pd.DataFrame({
    s.upper(): [split_stats[s]["class_counts"][c] for c in CLASS_NAMES]
    for s in ["train", "val", "test"]
}, index=CLASS_NAMES)

ax = class_dist_df.plot(kind="bar", figsize=(10, 4.5), colormap="viridis", edgecolor="black", alpha=0.88)
plt.title("Object Instance Distribution Across Train / Val / Test Splits", fontsize=12, fontweight="bold")
plt.xlabel("PPE Class")
plt.ylabel("Number of Annotated Bounding Boxes")
plt.xticks(rotation=20)
plt.grid(axis="y", linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()

## 4. YOLOv11 Model Architecture & Implementation

### Key Architectural Innovations of Ultralytics YOLOv11
YOLOv11 is a single-stage CNN object detector engineered for high accuracy-to-latency efficiency across edge and cloud GPUs:
1. **Backbone (Feature Extraction):**
   * **C3k2 Blocks:** Replaces YOLOv8's C2f blocks with faster, parameter-efficient Cross-Stage Partial blocks using smaller `3×3` kernels (`k=2` internal convolutions), improving feature representation with ~22% fewer parameters at comparable mAP.
   * **SPPF (Spatial Pyramid Pooling – Fast):** Chains sequential `5×5` max-pooling layers to expand the effective receptive field at minimal compute cost.
   * **C2PSA (Cross-Stage Partial with Position-Sensitive Attention):** Inserted after SPPF to capture long-range spatial dependencies and focus feature maps on small or partially occluded objects (such as medical gloves and goggles).
2. **Neck (Multi-Scale Feature Fusion):**
   * **FPN + PAN (Feature Pyramid Network + Path Aggregation Network):** Fuses high-level semantic features (top-down) with fine-grained spatial localization features (bottom-up) across three detection scales: **P3/8** (small objects), **P4/16** (medium objects), and **P5/32** (large objects).
3. **Head (Anchor-Free Decoupled Detection Head):**
   * Separates classification and bounding-box regression branches, utilizing **depthwise separable convolutions** in the classification branch to reduce FLOPs.
   * Trained with **Task-Aligned Assigner (TAL)**, **CIoU Loss** + **Distribution Focal Loss (DFL)** for box localization, and **Binary Cross-Entropy (BCE)** for multi-class classification.

We load pretrained COCO weights (`yolo11n.pt`) for transfer learning onto our 5-class CPPE-5 dataset.

In [ ]:
# Choose YOLOv11 variant: 'yolo11n.pt' (fast, ~2.6M params) or 'yolo11s.pt' (higher accuracy, ~9.4M params)
MODEL_WEIGHTS = "yolo11n.pt"

model = YOLO(MODEL_WEIGHTS)
print(f"Loaded pretrained YOLOv11 architecture: {MODEL_WEIGHTS}")

# Print detailed architectural summary (layers, parameters, gradients, GFLOPs)
info = model.info(detailed=False, verbose=True)

## 5. Model Training
We fine-tune `YOLOv11` on our prepared `cppe5_yolo/data.yaml` dataset:
* **Epochs:** `40` (with `patience=15` early stopping so training halts automatically if validation mAP plateaus; runs in ~3–5 minutes on a Colab T4 GPU).
* **Optimizer & Schedule:** Automatic optimizer selection (`AdamW` / `SGD`) with cosine learning-rate decay (`cos_lr=True`).
* **Online Data Augmentation:** Mosaic (disabled in final 10 epochs via `close_mosaic=10`), horizontal flip, HSV jitter, scale & translation augmentations.

In [ ]:
# Training Hyperparameters
EPOCHS = 40          # Increase to 60-80 for slightly higher mAP if you have extra GPU time
BATCH_SIZE = 16      # Fits comfortably on Colab T4 (15GB VRAM)
IMG_SIZE = 640
PROJECT_DIR = str(DATASET_DIR.parent / "runs_cv_task09")
RUN_NAME = "yolo11n_cppe5"

train_start_time = time.time()

train_results = model.train(
    data=str(YAML_PATH),
    epochs=EPOCHS,
    imgsz=IMG_SIZE,
    batch=BATCH_SIZE,
    device=DEVICE,
    project=PROJECT_DIR,
    name=RUN_NAME,
    exist_ok=True,
    patience=15,
    cos_lr=True,
    close_mosaic=10,
    seed=SEED,
    plots=True,
    verbose=True
)

train_duration_sec = time.time() - train_start_time
SAVE_DIR = Path(train_results.save_dir)
BEST_WEIGHTS_PATH = SAVE_DIR / "weights" / "best.pt"

print(f"\nTraining finished in {train_duration_sec / 60:.2f} minutes.")
print(f"Best trained weights saved at: {BEST_WEIGHTS_PATH}")

In [ ]:
# Visualize Training & Validation Curves (Losses, Precision, Recall, mAP@50, mAP@50-95)
results_csv_path = SAVE_DIR / "results.csv"
history_df = pd.read_csv(results_csv_path)
history_df.columns = [c.strip() for c in history_df.columns]

fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# 1. Box & DFL Loss
axes[0, 0].plot(history_df["epoch"], history_df["train/box_loss"], label="Train Box Loss", linewidth=2)
axes[0, 0].plot(history_df["epoch"], history_df["val/box_loss"], label="Val Box Loss", linewidth=2, linestyle="--")
axes[0, 0].set_title("Bounding Box Localization Loss (CIoU)")
axes[0, 0].set_xlabel("Epoch")
axes[0, 0].set_ylabel("Loss")
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# 2. Classification Loss
axes[0, 1].plot(history_df["epoch"], history_df["train/cls_loss"], label="Train Cls Loss", linewidth=2)
axes[0, 1].plot(history_df["epoch"], history_df["val/cls_loss"], label="Val Cls Loss", linewidth=2, linestyle="--")
axes[0, 1].set_title("Classification Loss (BCE)")
axes[0, 1].set_xlabel("Epoch")
axes[0, 1].set_ylabel("Loss")
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# 3. Precision & Recall
axes[1, 0].plot(history_df["epoch"], history_df["metrics/precision(B)"], label="Precision", linewidth=2, color="green")
axes[1, 0].plot(history_df["epoch"], history_df["metrics/recall(B)"], label="Recall", linewidth=2, color="orange")
axes[1, 0].set_title("Validation Precision & Recall")
axes[1, 0].set_xlabel("Epoch")
axes[1, 0].set_ylabel("Score")
axes[1, 0].set_ylim(0, 1.05)
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# 4. mAP@50 and mAP@50-95
axes[1, 1].plot(history_df["epoch"], history_df["metrics/mAP50(B)"], label="mAP@0.50", linewidth=2.5, color="royalblue")
axes[1, 1].plot(history_df["epoch"], history_df["metrics/mAP50-95(B)"], label="mAP@0.50:0.95", linewidth=2.5, color="crimson")
axes[1, 1].set_title("Validation Mean Average Precision (mAP)")
axes[1, 1].set_xlabel("Epoch")
axes[1, 1].set_ylabel("mAP")
axes[1, 1].set_ylim(0, 1.05)
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

plt.suptitle("YOLOv11 Training Progression on CPPE-5", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig(SAVE_DIR / "custom_training_curves.png", dpi=150)
plt.show()

# Display Ultralytics generated Confusion Matrix & PR Curve if present
for plot_name, title in [("confusion_matrix_normalized.png", "Normalized Confusion Matrix"), ("BoxPR_curve.png", "Precision-Recall Curve"), ("PR_curve.png", "Precision-Recall Curve")]:
    p = SAVE_DIR / plot_name
    if p.exists():
        plt.figure(figsize=(8, 6))
        plt.imshow(cv2.cvtColor(cv2.imread(str(p)), cv2.COLOR_BGR2RGB))
        plt.title(title, fontsize=12, fontweight="bold")
        plt.axis("off")
        plt.show()

## 6. Model Evaluation: mAP, Explicit IoU, and Inference Speed

We evaluate our best trained checkpoint (`best.pt`) on both the **Validation** split and the **Held-Out Test** split using all three metrics specified in the assignment:
1. **mAP (mean Average Precision):** `mAP@0.50`, `mAP@0.75`, and COCO `mAP@0.50:0.95`, plus per-class AP, Precision, Recall, and F1-score.
2. **IoU (Intersection over Union):** Ultralytics reports mAP across IoU thresholds but does not directly print average box-level IoU. Below, we implement an exact **IoU evaluation function** matching predicted boxes to ground-truth boxes of the same class (`IoU >= 0.50`) to measure **Mean IoU**, **Median IoU**, per-class IoU, and plot the IoU distribution histogram.
3. **Inference Speed:** Breakdown of preprocess, forward-pass inference, and NMS postprocess latency (`ms/image`), plus a warmed-up real-world **FPS (Frames Per Second)** benchmark.

In [ ]:
# Load the best saved weights
best_model = YOLO(str(BEST_WEIGHTS_PATH))

# 1. Official Ultralytics Validation on Val and Held-Out Test Splits
val_metrics = best_model.val(data=str(YAML_PATH), split="val", imgsz=IMG_SIZE, device=DEVICE, verbose=False)
test_metrics = best_model.val(data=str(YAML_PATH), split="test", imgsz=IMG_SIZE, device=DEVICE, verbose=False)

def summarize_ultralytics_metrics(m, split_label):
    p = float(m.box.mp)
    r = float(m.box.mr)
    f1 = 2 * p * r / (p + r + 1e-9)
    total_ms = m.speed["preprocess"] + m.speed["inference"] + m.speed["postprocess"]
    return {
        "Split": split_label,
        "Precision": round(p, 4),
        "Recall": round(r, 4),
        "F1-Score": round(f1, 4),
        "mAP@0.50": round(float(m.box.map50), 4),
        "mAP@0.75": round(float(m.box.map75), 4),
        "mAP@0.50:0.95": round(float(m.box.map), 4),
        "Inference (ms/img)": round(float(m.speed["inference"]), 2),
        "Total Latency (ms/img)": round(float(total_ms), 2),
        "Approx FPS": round(1000.0 / max(total_ms, 1e-6), 1),
    }

overall_eval_df = pd.DataFrame([
    summarize_ultralytics_metrics(val_metrics, "Validation"),
    summarize_ultralytics_metrics(test_metrics, "Unseen Test")
])
print("=== Overall Detection Performance (mAP & Speed) ===")
display(overall_eval_df)

# Per-Class AP Breakdown on the Unseen Test Set
per_class_rows = []
for cls_idx, cls_name in enumerate(CLASS_NAMES):
    try:
        p_c, r_c, ap50_c, ap_c = test_metrics.box.class_result(cls_idx)
        per_class_rows.append({
            "Class": cls_name,
            "Precision": round(float(p_c), 4),
            "Recall": round(float(r_c), 4),
            "AP@0.50": round(float(ap50_c), 4),
            "AP@0.50:0.95": round(float(ap_c), 4)
        })
    except Exception:
        pass

per_class_df = pd.DataFrame(per_class_rows)
print("\n=== Per-Class Metrics on Unseen Test Split ===")
display(per_class_df)

In [ ]:
# 2. Explicit Box-Level Intersection over Union (IoU) Computation on Unseen Test Set
def compute_iou_matrix(boxes_a: np.ndarray, boxes_b: np.ndarray) -> np.ndarray:
    """
    Compute pairwise IoU between boxes_a (N, 4) and boxes_b (M, 4) in [x1, y1, x2, y2] format.
    """
    if len(boxes_a) == 0 or len(boxes_b) == 0:
        return np.zeros((len(boxes_a), len(boxes_b)), dtype=np.float32)

    inter_x1 = np.maximum(boxes_a[:, None, 0], boxes_b[None, :, 0])
    inter_y1 = np.maximum(boxes_a[:, None, 1], boxes_b[None, :, 1])
    inter_x2 = np.minimum(boxes_a[:, None, 2], boxes_b[None, :, 2])
    inter_y2 = np.minimum(boxes_a[:, None, 3], boxes_b[None, :, 3])

    inter_w = np.maximum(0.0, inter_x2 - inter_x1)
    inter_h = np.maximum(0.0, inter_y2 - inter_y1)
    inter_area = inter_w * inter_h

    area_a = np.maximum(0.0, boxes_a[:, 2] - boxes_a[:, 0]) * np.maximum(0.0, boxes_a[:, 3] - boxes_a[:, 1])
    area_b = np.maximum(0.0, boxes_b[:, 2] - boxes_b[:, 0]) * np.maximum(0.0, boxes_b[:, 3] - boxes_b[:, 1])

    union_area = area_a[:, None] + area_b[None, :] - inter_area + 1e-9
    return inter_area / union_area


def evaluate_exact_iou_and_speed(yolo_model, img_dir: Path, lbl_dir: Path, conf_thresh: float = 0.25, iou_match_thresh: float = 0.50):
    img_paths = sorted(glob.glob(str(img_dir / "*.jpg")))
    matched_ious = []
    per_class_ious = {c: [] for c in CLASS_NAMES}
    inference_times_ms = []

    # Warm-up pass (prevents CUDA initialization overhead from skewing FPS)
    if img_paths:
        _ = yolo_model.predict(img_paths[0], imgsz=IMG_SIZE, conf=conf_thresh, device=DEVICE, verbose=False)

    for img_path in img_paths:
        img = cv2.imread(img_path)
        h, w = img.shape[:2]

        # Load ground-truth boxes in [x1, y1, x2, y2]
        lbl_path = lbl_dir / (Path(img_path).stem + ".txt")
        gt_boxes, gt_classes = [], []
        if lbl_path.exists():
            with open(lbl_path, "r") as f:
                for line in f:
                    c, xc, yc, bw, bh = map(float, line.strip().split())
                    x1 = (xc - bw / 2.0) * w
                    y1 = (yc - bh / 2.0) * h
                    x2 = (xc + bw / 2.0) * w
                    y2 = (yc + bh / 2.0) * h
                    gt_boxes.append([x1, y1, x2, y2])
                    gt_classes.append(int(c))

        gt_boxes = np.array(gt_boxes, dtype=np.float32) if gt_boxes else np.zeros((0, 4), dtype=np.float32)
        gt_classes = np.array(gt_classes, dtype=np.int32) if gt_classes else np.zeros((0,), dtype=np.int32)

        # Timed prediction
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        t0 = time.perf_counter()
        res = yolo_model.predict(img, imgsz=IMG_SIZE, conf=conf_thresh, device=DEVICE, verbose=False)[0]
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        t1 = time.perf_counter()
        inference_times_ms.append((t1 - t0) * 1000.0)

        pred_boxes = res.boxes.xyxy.cpu().numpy() if len(res.boxes) > 0 else np.zeros((0, 4), dtype=np.float32)
        pred_classes = res.boxes.cls.cpu().numpy().astype(np.int32) if len(res.boxes) > 0 else np.zeros((0,), dtype=np.int32)
        pred_confs = res.boxes.conf.cpu().numpy() if len(res.boxes) > 0 else np.zeros((0,), dtype=np.float32)

        # Greedy matching by highest confidence within each class
        order = np.argsort(-pred_confs)
        used_gt = set()
        for p_idx in order:
            p_cls = pred_classes[p_idx]
            p_box = pred_boxes[p_idx:p_idx + 1]
            same_cls_gt = [g for g in range(len(gt_boxes)) if gt_classes[g] == p_cls and g not in used_gt]
            if not same_cls_gt:
                continue
            ious = compute_iou_matrix(p_box, gt_boxes[same_cls_gt])[0]
            best_local = int(np.argmax(ious))
            best_iou = float(ious[best_local])
            if best_iou >= iou_match_thresh:
                used_gt.add(same_cls_gt[best_local])
                matched_ious.append(best_iou)
                per_class_ious[CLASS_NAMES[p_cls]].append(best_iou)

    return matched_ious, per_class_ious, inference_times_ms


matched_ious, per_class_ious, timed_ms = evaluate_exact_iou_and_speed(
    best_model, DATASET_DIR / "images/test", DATASET_DIR / "labels/test", conf_thresh=0.25, iou_match_thresh=0.50
)

mean_iou = float(np.mean(matched_ious)) if matched_ious else 0.0
median_iou = float(np.median(matched_ious)) if matched_ious else 0.0
avg_latency_ms = float(np.mean(timed_ms)) if timed_ms else 0.0
real_fps = 1000.0 / max(avg_latency_ms, 1e-6)

print("=== Explicit IoU & End-to-End Inference Speed (Unseen Test Split) ===")
print(f"Matched True-Positive Detections (IoU >= 0.50) : {len(matched_ious)}")
print(f"Mean IoU (Localization Tightness)              : {mean_iou:.4f}")
print(f"Median IoU                                     : {median_iou:.4f}")
print(f"End-to-End Inference Latency (Mean)            : {avg_latency_ms:.2f} ms/frame")
print(f"End-to-End Throughput                          : {real_fps:.1f} FPS")

iou_class_df = pd.DataFrame([
    {
        "Class": c,
        "Matched TPs": len(vals),
        "Mean IoU": round(float(np.mean(vals)), 4) if vals else 0.0,
        "Median IoU": round(float(np.median(vals)), 4) if vals else 0.0
    }
    for c, vals in per_class_ious.items()
])
display(iou_class_df)

# Plot IoU Distribution & Latency Distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
axes[0].hist(matched_ious, bins=20, color="teal", edgecolor="black", alpha=0.8)
axes[0].axvline(mean_iou, color="crimson", linestyle="--", linewidth=2, label=f"Mean IoU = {mean_iou:.3f}")
axes[0].set_title("Distribution of Matched Bounding-Box IoU (Unseen Test Set)")
axes[0].set_xlabel("Intersection over Union (IoU)")
axes[0].set_ylabel("Count")
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].hist(timed_ms, bins=20, color="slateblue", edgecolor="black", alpha=0.8)
axes[1].axvline(avg_latency_ms, color="orange", linestyle="--", linewidth=2, label=f"Mean = {avg_latency_ms:.1f} ms ({real_fps:.1f} FPS)")
axes[1].set_title("Per-Frame End-to-End Inference Latency Distribution")
axes[1].set_xlabel("Latency (ms/frame)")
axes[1].set_ylabel("Count")
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(SAVE_DIR / "iou_and_speed_analysis.png", dpi=150)
plt.show()

## 7. Testing on Unseen Data (Ground Truth vs. Predictions)
To verify generalization beyond the training and validation splits, we run `best.pt` on completely unseen held-out test images and display **Ground Truth** side by side with **YOLOv11 Predictions**.

In [ ]:
# Side-by-side comparison of Ground Truth vs. YOLOv11 Predictions on unseen test images
test_img_paths = sorted(glob.glob(str(DATASET_DIR / "images/test/*.jpg")))
selected_test_imgs = random.sample(test_img_paths, min(6, len(test_img_paths)))

fig, axes = plt.subplots(len(selected_test_imgs), 2, figsize=(14, 5 * len(selected_test_imgs)))
if len(selected_test_imgs) == 1:
    axes = np.array([axes])

for row_idx, img_path in enumerate(selected_test_imgs):
    img_bgr = cv2.imread(img_path)
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    h, w = img_rgb.shape[:2]

    # Left column: Ground Truth
    ax_gt = axes[row_idx, 0]
    ax_gt.imshow(img_rgb)
    lbl_path = DATASET_DIR / "labels/test" / (Path(img_path).stem + ".txt")
    gt_count = 0
    if lbl_path.exists():
        with open(lbl_path, "r") as f:
            for line in f:
                c, xc, yc, bw, bh = map(float, line.strip().split())
                c = int(c)
                x1, y1 = (xc - bw / 2.0) * w, (yc - bh / 2.0) * h
                rect = patches.Rectangle((x1, y1), bw * w, bh * h, linewidth=2, edgecolor=COLORS[c], facecolor="none")
                ax_gt.add_patch(rect)
                ax_gt.text(
                    x1, max(0, y1 - 4), CLASS_NAMES[c],
                    color="white", fontsize=8, fontweight="bold",
                    bbox=dict(facecolor=COLORS[c], alpha=0.85, pad=1, edgecolor="none")
                )
                gt_count += 1
    ax_gt.set_title(f"Ground Truth ({gt_count} objs) — {Path(img_path).name}", fontsize=11, fontweight="bold")
    ax_gt.axis("off")

    # Right column: YOLOv11 Prediction
    res = best_model.predict(img_bgr, imgsz=IMG_SIZE, conf=0.25, iou=0.45, device=DEVICE, verbose=False)[0]
    pred_annotated_rgb = cv2.cvtColor(res.plot(), cv2.COLOR_BGR2RGB)

    ax_pred = axes[row_idx, 1]
    ax_pred.imshow(pred_annotated_rgb)
    ax_pred.set_title(f"YOLOv11 Prediction ({len(res.boxes)} detections)", fontsize=11, fontweight="bold")
    ax_pred.axis("off")

plt.suptitle("Unseen Test Set Generalization: Ground Truth vs. YOLOv11 Predictions", fontsize=14, fontweight="bold", y=1.002)
plt.tight_layout()
plt.savefig(SAVE_DIR / "unseen_test_predictions_grid.png", dpi=150, bbox_inches="tight")
plt.show()

## 8. Live Prediction on Frames (Real-Time Stream + Colab Webcam + Local Script)

Because Google Colab executes in a cloud container without direct OS access to `/dev/video0`, this section provides **three complete live-frame prediction workflows**:
* **Mode A (Automated Live Frame Stream — runs immediately in Colab):** Generates a continuous test frame stream (or accepts any uploaded `.mp4`), runs real-time frame-by-frame YOLOv11 inference with a live **FPS + Object Count HUD**, saves the annotated video, and plays it inline.
* **Mode B (Interactive Google Colab Live Webcam):** Uses HTML5 WebRTC + JavaScript (`google.colab.output.eval_js`) to capture live frames directly from your laptop webcam inside the browser and overlay YOLOv11 detections.
* **Mode C (Standalone Local Webcam Script):** Writes `live_webcam_yolo11.py` so you can run `python live_webcam_yolo11.py` with `cv2.VideoCapture(0)` at 30+ FPS on your own machine using the exported `best.pt` weights.

In [ ]:
# === MODE A: Automated Live Frame-by-Frame Stream Detection (Works out-of-the-box in Colab) ===
from IPython.display import HTML, display as ipy_display
from base64 import b64encode

# Build a synthetic multi-frame test video stream from unseen test images (simulates live camera frames)
DEMO_INPUT_VIDEO = SAVE_DIR / "live_input_stream.mp4"
DEMO_OUTPUT_VIDEO = SAVE_DIR / "live_predicted_stream.mp4"

stream_imgs = sorted(glob.glob(str(DATASET_DIR / "images/test/*.jpg")))[:15]
fps_in = 10
fourcc = cv2.VideoWriter_fourcc(*"mp4v")
writer_in = cv2.VideoWriter(str(DEMO_INPUT_VIDEO), fourcc, fps_in, (IMG_SIZE, IMG_SIZE))

# Hold each unseen frame for 10 frames (1 second per scene) with slight pan/zoom motion to simulate a live camera feed
for img_p in stream_imgs:
    frame = cv2.imread(img_p)
    for step in range(10):
        # Mild zoom/crop jitter to create continuous frame motion
        margin = int(step * 2)
        cropped = frame[margin:IMG_SIZE - margin, margin:IMG_SIZE - margin] if margin > 0 else frame
        motion_frame = cv2.resize(cropped, (IMG_SIZE, IMG_SIZE))
        writer_in.write(motion_frame)
writer_in.release()

# Run real-time frame-by-frame live prediction loop
cap = cv2.VideoCapture(str(DEMO_INPUT_VIDEO))
writer_out = cv2.VideoWriter(str(DEMO_OUTPUT_VIDEO), fourcc, fps_in, (IMG_SIZE, IMG_SIZE))

frame_count = 0
live_fps_list = []
sample_live_frames = []

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    t_start = time.perf_counter()
    results = best_model.predict(frame, imgsz=IMG_SIZE, conf=0.30, device=DEVICE, verbose=False)
    annotated = results[0].plot()
    elapsed = time.perf_counter() - t_start
    inst_fps = 1.0 / max(elapsed, 1e-6)
    live_fps_list.append(inst_fps)

    # Overlay real-time HUD (FPS, Frame #, Active Detections)
    n_det = len(results[0].boxes)
    cv2.rectangle(annotated, (8, 8), (340, 72), (20, 20, 20), -1)
    cv2.putText(annotated, f"YOLOv11 LIVE | FPS: {inst_fps:5.1f}", (16, 34),
                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 120), 2, cv2.LINE_AA)
    cv2.putText(annotated, f"Frame: {frame_count:03d} | Objects: {n_det}", (16, 60),
                cv2.FONT_HERSHEY_SIMPLEX, 0.65, (255, 255, 255), 2, cv2.LINE_AA)

    writer_out.write(annotated)
    if frame_count % 30 == 0 and len(sample_live_frames) < 4:
        sample_live_frames.append(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
    frame_count += 1

cap.release()
writer_out.release()

print(f"Processed {frame_count} live stream frames.")
print(f"Average Live Stream Inference Speed: {np.mean(live_fps_list):.1f} FPS")
print(f"Annotated output video saved to: {DEMO_OUTPUT_VIDEO}")

# Display 4 key snapshots from the live stream
fig, axes = plt.subplots(1, len(sample_live_frames), figsize=(20, 5))
if len(sample_live_frames) == 1:
    axes = [axes]
for ax, snap in zip(axes, sample_live_frames):
    ax.imshow(snap)
    ax.axis("off")
plt.suptitle("Snapshots from Live Frame-by-Frame YOLOv11 Prediction Stream", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(SAVE_DIR / "live_stream_snapshots.png", dpi=150)
plt.show()

In [ ]:
# === MODE B: Interactive Live Webcam Capture & YOLOv11 Prediction inside Google Colab ===
# Run this cell in Google Colab to capture a live frame from your webcam and run YOLOv11 detection on it!
def run_colab_webcam_detection(model_obj, conf_thresh=0.25):
    try:
        from google.colab.output import eval_js
        from IPython.display import display as colab_display, Javascript
        import base64
    except ImportError:
        print("Google Colab environment not detected. Use Mode C (live_webcam_yolo11.py) on your local machine.")
        return

    js = Javascript('''
    async function captureFrame() {
      const div = document.createElement('div');
      const captureBtn = document.createElement('button');
      captureBtn.textContent = 'Capture Live Frame for YOLOv11 Detection';
      captureBtn.style.cssText = 'padding:10px 18px;font-size:15px;font-weight:bold;background:#1a73e8;color:white;border:none;border-radius:6px;cursor:pointer;margin-bottom:8px;';
      div.appendChild(captureBtn);

      const video = document.createElement('video');
      video.style.display = 'block';
      video.style.borderRadius = '8px';
      const stream = await navigator.mediaDevices.getUserMedia({video: true});
      document.body.appendChild(div);
      div.appendChild(video);
      video.srcObject = stream;
      await video.play();

      google.colab.output.setIframeHeight(document.documentElement.scrollHeight, true);
      await new Promise((resolve) => captureBtn.onclick = resolve);

      const canvas = document.createElement('canvas');
      canvas.width = video.videoWidth;
      canvas.height = video.videoHeight;
      canvas.getContext('2d').drawImage(video, 0, 0);
      stream.getVideoTracks()[0].stop();
      div.remove();
      return canvas.toDataURL('image/jpeg', 0.9);
    }
    ''')
    colab_display(js)
    data = eval_js('captureFrame()')
    binary = base64.b64decode(data.split(',')[1])
    img_arr = np.frombuffer(binary, dtype=np.uint8)
    frame_bgr = cv2.imdecode(img_arr, cv2.IMREAD_COLOR)

    t0 = time.perf_counter()
    res = model_obj.predict(frame_bgr, imgsz=IMG_SIZE, conf=conf_thresh, device=DEVICE, verbose=False)[0]
    dt_ms = (time.perf_counter() - t0) * 1000.0
    annotated_rgb = cv2.cvtColor(res.plot(), cv2.COLOR_BGR2RGB)

    plt.figure(figsize=(10, 7))
    plt.imshow(annotated_rgb)
    plt.title(f"Colab Live Webcam Detection | {len(res.boxes)} objects | {dt_ms:.1f} ms ({1000/dt_ms:.1f} FPS)", fontsize=12, fontweight="bold")
    plt.axis("off")
    plt.show()

# Uncomment the line below when running interactively in Google Colab with a webcam attached:
# run_colab_webcam_detection(best_model, conf_thresh=0.25)
print("Interactive Colab webcam function ready. Uncomment `run_colab_webcam_detection(best_model)` above to test with your webcam.")

In [ ]:
# === MODE C: Export Standalone Local Real-Time Webcam Script (live_webcam_yolo11.py) ===
local_webcam_script = '''"""
Real-Time Local Webcam Object Detection using Trained YOLOv11 Model.
Usage:
    pip install ultralytics opencv-python
    python live_webcam_yolo11.py --weights best.pt --source 0 --conf 0.30
"""
import argparse
import time
import cv2
from ultralytics import YOLO


def main():
    parser = argparse.ArgumentParser(description="Live YOLOv11 Webcam Object Detection")
    parser.add_argument("--weights", type=str, default="best.pt", help="Path to trained YOLOv11 weights")
    parser.add_argument("--source", type=str, default="0", help="Webcam index (0) or video file path")
    parser.add_argument("--conf", type=float, default=0.30, help="Confidence threshold")
    parser.add_argument("--imgsz", type=int, default=640, help="Inference image size")
    args = parser.parse_args()

    model = YOLO(args.weights)
    source = int(args.source) if args.source.isdigit() else args.source
    cap = cv2.VideoCapture(source)

    if not cap.isOpened():
        raise RuntimeError(f"Could not open video source: {args.source}")

    print("Live YOLOv11 detection started. Press 'q' to quit.")
    while True:
        ret, frame = cap.read()
        if not ret:
            break

        t0 = time.perf_counter()
        results = model.predict(frame, imgsz=args.imgsz, conf=args.conf, verbose=False)
        annotated = results[0].plot()
        fps = 1.0 / max(time.perf_counter() - t0, 1e-6)

        cv2.putText(annotated, f"YOLOv11 Live | FPS: {fps:.1f} | Objs: {len(results[0].boxes)}",
                    (15, 35), cv2.FONT_HERSHEY_SIMPLEX, 0.85, (0, 255, 100), 2, cv2.LINE_AA)
        cv2.imshow("YOLOv11 Live Object Detection", annotated)

        if cv2.waitKey(1) & 0xFF == ord("q"):
            break

    cap.release()
    cv2.destroyAllWindows()


if __name__ == "__main__":
    main()
'''

LOCAL_SCRIPT_PATH = SAVE_DIR / "live_webcam_yolo11.py"
LOCAL_SCRIPT_PATH.write_text(local_webcam_script)
print(f"Saved local live-webcam inference script to: {LOCAL_SCRIPT_PATH}")

## 9. Automated Project Report & Deliverables Export
This final section automatically compiles a structured **Markdown documentation report** (`CV_Task_09_Report.md`) containing your exact trained model metrics, architectural explanation, training configuration, and visual references, and bundles everything (`best.pt`, `CV_Task_09_Report.md`, `live_webcam_yolo11.py`, evaluation charts, and annotated live video) into a downloadable `.zip` archive.

In [ ]:
# Auto-generate comprehensive Task Report populated with this run's actual metrics
val_row = overall_eval_df.iloc[0]
test_row = overall_eval_df.iloc[1]

report_md = f"""# Object Detection Using CNN Architecture (YOLOv11) — Technical Report

## 1. Objective
Develop, train, evaluate, and deploy an end-to-end CNN-based object detection pipeline using the **Ultralytics YOLOv11** architecture (`{MODEL_WEIGHTS}`) capable of simultaneous multi-object classification and bounding-box localization on both static unseen images and live video frames.

---

## 2. Dataset & Preprocessing Pipeline
* **Dataset:** CPPE-5 (Medical Personal Protective Equipment — `rishitdagli/cppe-5`)
* **Classes ({NUM_CLASSES}):** {", ".join([f"`{c}`" for c in CLASS_NAMES])}
* **Dataset Splits (Stratified 75% / 15% / 10%):**
  * **Train:** {split_stats['train']['images']} images ({split_stats['train']['boxes']} annotated objects)
  * **Validation:** {split_stats['val']['images']} images ({split_stats['val']['boxes']} annotated objects)
  * **Unseen Test:** {split_stats['test']['images']} images ({split_stats['test']['boxes']} annotated objects)
* **Preprocessing Applied:**
  1. RGB standardization and **aspect-ratio-preserving letterbox resizing** to `{IMG_SIZE}×{IMG_SIZE}` (neutral gray padding `(114, 114, 114)`).
  2. Pixel normalization verification to `[0.0, 1.0]` (`Mean RGB = [{mean_rgb[0]:.3f}, {mean_rgb[1]:.3f}, {mean_rgb[2]:.3f}]`, `Std RGB = [{std_rgb[0]:.3f}, {std_rgb[1]:.3f}, {std_rgb[2]:.3f}]`).
  3. Conversion of raw COCO pixel boxes `[x_min, y_min, w, h]` into normalized YOLO labels `[class_id, x_center, y_center, width, height]` with strict boundary clamping and degenerate-box filtering.

---

## 3. YOLOv11 Model Architecture
YOLOv11 is a single-stage anchor-free CNN detector comprising three core stages:
1. **Backbone:** Uses **C3k2** (Cross-Stage Partial blocks with lightweight `3×3` convolutions), **SPPF** (Spatial Pyramid Pooling – Fast), and **C2PSA** (Position-Sensitive Attention) to capture both fine local textures and global spatial context.
2. **Neck:** **FPN + PAN** bidirectional feature pyramid fusing multi-scale feature maps (`P3/8`, `P4/16`, `P5/32`) for small, medium, and large object detection.
3. **Head:** **Decoupled anchor-free detection head** with depthwise separable convolutions for classification and Distribution Focal Loss (DFL) + CIoU regression for precise bounding-box coordinates.

---

## 4. Training Process & Configuration
* **Pretrained Base Weights:** `{MODEL_WEIGHTS}` (COCO transfer learning)
* **Epochs:** `{EPOCHS}` (with `patience=15` early stopping)
* **Image Size:** `{IMG_SIZE}×{IMG_SIZE}` | **Batch Size:** `{BATCH_SIZE}` | **LR Schedule:** Cosine Annealing (`cos_lr=True`)
* **Augmentations:** Mosaic (disabled last 10 epochs), HSV color jitter, random horizontal flip, scale & translation.
* **Total Training Time:** `{train_duration_sec / 60:.2f}` minutes on `{torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}`.

---

## 5. Quantitative Evaluation Metrics

### 5.1 Overall Performance (Validation vs. Held-Out Unseen Test)
| Split | Precision | Recall | F1-Score | mAP@0.50 | mAP@0.75 | mAP@0.50:0.95 | Inference (ms/img) | Approx FPS |
| :--- | :---: | :---: | :---: | :---: | :---: | :---: | :---: | :---: |
| **Validation** | {val_row['Precision']} | {val_row['Recall']} | {val_row['F1-Score']} | {val_row['mAP@0.50']} | {val_row['mAP@0.75']} | {val_row['mAP@0.50:0.95']} | {val_row['Inference (ms/img)']} | {val_row['Approx FPS']} |
| **Unseen Test** | {test_row['Precision']} | {test_row['Recall']} | {test_row['F1-Score']} | {test_row['mAP@0.50']} | {test_row['mAP@0.75']} | {test_row['mAP@0.50:0.95']} | {test_row['Inference (ms/img)']} | {test_row['Approx FPS']} |

### 5.2 Box-Level Localization (IoU) & Live Speed Benchmark (Unseen Test Set)
* **Mean IoU (Matched True Positives, IoU ≥ 0.50):** `{mean_iou:.4f}`
* **Median IoU:** `{median_iou:.4f}`
* **End-to-End Frame Latency:** `{avg_latency_ms:.2f} ms/frame` (`{real_fps:.1f} FPS`)
* **Live Video Stream Throughput:** `{np.mean(live_fps_list):.1f} FPS` across `{frame_count}` frames

---

## 6. Generated Visualizations & Deliverables
* `weights/best.pt` — Trained YOLOv11 object detection model weights
* `custom_training_curves.png` — Training/validation loss and mAP curves
* `iou_and_speed_analysis.png` — Exact IoU histogram and per-frame latency distribution
* `unseen_test_predictions_grid.png` — Side-by-side Ground Truth vs. YOLOv11 unseen test predictions
* `live_stream_snapshots.png` & `live_predicted_stream.mp4` — Real-time frame prediction outputs
* `live_webcam_yolo11.py` — Standalone OpenCV real-time webcam inference script
"""

REPORT_PATH = SAVE_DIR / "CV_Task_09_Report.md"
REPORT_PATH.write_text(report_md)
print(f"Report generated successfully: {REPORT_PATH}\n")
print(report_md)

In [ ]:
# Bundle all deliverables into a single downloadable ZIP file for submission
BUNDLE_DIR = SAVE_DIR / "submission_bundle"
BUNDLE_DIR.mkdir(exist_ok=True)

files_to_bundle = [
    BEST_WEIGHTS_PATH,
    REPORT_PATH,
    LOCAL_SCRIPT_PATH,
    SAVE_DIR / "custom_training_curves.png",
    SAVE_DIR / "iou_and_speed_analysis.png",
    SAVE_DIR / "unseen_test_predictions_grid.png",
    SAVE_DIR / "live_stream_snapshots.png",
    SAVE_DIR / "live_predicted_stream.mp4",
    SAVE_DIR / "confusion_matrix_normalized.png",
    SAVE_DIR / "results.png",
    SAVE_DIR / "results.csv",
]

for fpath in files_to_bundle:
    if fpath.exists():
        shutil.copy2(fpath, BUNDLE_DIR / fpath.name)

ZIP_BASE = str(SAVE_DIR.parent / "CV_Task_09_YOLOv11_Deliverables")
ZIP_PATH = shutil.make_archive(ZIP_BASE, "zip", BUNDLE_DIR)
print(f"Submission bundle created: {ZIP_PATH}")

# Trigger automatic download in Google Colab
try:
    from google.colab import files
    files.download(ZIP_PATH)
except ImportError:
    print("Not running inside Google Colab; locate your submission bundle at:", ZIP_PATH)